# COMP8240 · SanText final report: remaining runs (notebook A/B)

This notebook produces the results the final report still needs, in priority order:

| Tier | What runs | Why the report needs it | GPU time |
|---|---|---|---|
| **1** | Dreaddit and the review dataset at ε = 1, 2, 3 with **SanText and SanText+**; **SanText+ on SST-2**; reproducibility checks | Results on new data across ε, related to the replication; the paper's second mechanism | about 1.5 h |
| **2** | SST-2 SanText ε = 1, 2, 3 and the unsanitized baseline with **seeds 42, 43, 44** | Replication with run-to-run spread instead of single runs | about 1.5 h more |
| **3** | SanText+ on SST-2 with seeds 43–44; new datasets at ε = 3 with seeds 43–44 | Spread for the remaining settings | about 1 h more |

The CPU sanitizes in the background (the authors' unchanged `run_SanText.py`) while the GPU fine-tunes whatever is ready.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** If Colab says the notebook wasn't written by Google, choose **Run anyway**, and approve the **Google Drive** pop-up.
3. Keep this tab open and click into it every 20–30 minutes, so Colab doesn't disconnect.

**If Colab disconnects or you stop it:** just **Runtime → Run all** again. Everything is saved to `MyDrive/COMP8240_final` after each run, and finished runs are skipped. To stop after Tier 1 or 2, set `MAX_TIER` in Step 2.

**Your review dataset** is read from `MyDrive/COMP8240_update/new_data/app_reviews/`, where the 9 Oct notebook saved it. It is the only copy, so this notebook never changes it.

**When it finishes**, Step 4 downloads `final_runs_outputs.zip` (also saved to Drive). Upload it in the chat.

In [ ]:
# Step 0 · Setup and GPU check (about 2 minutes)
import os, sys, re, json, time, glob, shutil, subprocess, hashlib, platform
BASE = "/content/work"
for d in ("data", "logs", "results", "san", "analysis"):
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
os.chdir(BASE)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.")

from google.colab import drive
drive.mount("/content/drive")
OLD = "/content/drive/MyDrive/COMP8240_update"   # the 9 Oct run: your review dataset and earlier results
NEW = "/content/drive/MyDrive/COMP8240_final"    # everything this notebook produces
os.makedirs(NEW, exist_ok=True)


def backup(*paths):
    """Copy files or folders under BASE to the same place under NEW (Drive)."""
    for p in paths:
        if not os.path.exists(p):
            continue
        dest = os.path.join(NEW, os.path.relpath(p, BASE))
        if os.path.isdir(p):
            shutil.copytree(p, dest, dirs_exist_ok=True)
        else:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(p, dest)


# Resume: bring back whatever an earlier session of this notebook finished
restored = []
for sub in ("results", "san", "logs", "analysis", "data/glove.filtered.txt", "data/glove.filtered.txt.json"):
    src = os.path.join(NEW, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(BASE, sub), dirs_exist_ok=True)
        restored.append(sub)
    elif os.path.isfile(src):
        shutil.copy2(src, os.path.join(BASE, sub))
        restored.append(sub)
print("restored from Drive:", ", ".join(restored) or "nothing (first session)")

SANTEXT_COMMIT = "9c35f6e"   # the authors' latest commit (15 Mar 2022)
if not os.path.isdir(f"{BASE}/SanText"):
    !git clone -q https://github.com/xiangyue9607/SanText.git {BASE}/SanText
!cd {BASE}/SanText && git checkout -q {SANTEXT_COMMIT} && git log -1 --format="SanText code: commit %h (%cd)" --date=short

import multiprocessing, numpy, scipy, sklearn, spacy, transformers
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "spacy": spacy.__version__, "numpy": numpy.__version__, "scipy": scipy.__version__,
       "scikit-learn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0), "cpu_cores": os.cpu_count(),
       "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
       "multiprocessing_start_method": multiprocessing.get_start_method()}
with open(f"{BASE}/analysis/environment.json", "w") as f:
    json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))

### The scripts (written to disk; nothing to do here)
- `finetune_sanitized.py`: the fine-tuning script from 9 Oct, unchanged.
- `measure.py`: records wall time and peak memory of each sanitization, unchanged.
- `filter_glove.py`: copies the GloVe lines `run_SanText.py` can use into a small file, so each run loads in seconds. `run_SanText.py` gets the same words, vectors and probability matrix; on the authors' code this gave byte-identical SanText output.
- `sanitize_queue.py`: runs the sanitizations one after another in the background.
- `analyse_runs.py`: the summary tables (accuracy vs the paper, words left unchanged, reproducibility).
- `make_plan.py`: the run plan, in priority order.

In [ ]:
%%writefile finetune_sanitized.py
#!/usr/bin/env python3
"""
finetune_sanitized.py: fine-tune BERT on (sanitized) text and report dev-set utility.

Why this script exists
  SanText's released fine-tuning code (run_glue.py + dataset_glue.py) imports
  transformers 3.3.1 module paths such as `transformers.tokenization_bart`,
  which were removed in transformers 4.0, so it no longer runs on a current
  install. This script re-implements the same fine-tuning step with the paper's
  settings (Yue et al., 2021, Sec. 5.1, SST-2/QNLI):
      bert-base-uncased, max_seq_length 128, batch size 64, learning rate 2e-5,
      3 epochs, AdamW without weight decay, linear decay without warmup,
      gradient clipping 1.0, seed 42.
  One deliberate difference: each batch is padded to its longest example rather
  than to 128 tokens. Padding is masked out of attention, so the model computes
  the same thing, but short-sentence data such as SST-2 trains several times faster.

Input
  --data_dir must contain train.tsv and dev.tsv with a header row, as written by
  run_SanText.py (or the original GLUE files):
    single sentence (SST-2 style):  sentence <TAB> label
    sentence pair  (QNLI style):    index <TAB> text_a <TAB> text_b <TAB> label
  The format is detected from the header (2 columns = single, 4 or more = pair).

Output
  results/<run_name>/metrics.json         settings, environment, per-epoch and final dev metrics
  results/<run_name>/dev_predictions.tsv  gold and predicted label for every dev row

Examples
  # 1-minute smoke test (run this first on a new machine)
  python finetune_sanitized.py --data_dir data/SST-2 --max_train_samples 512 --epochs 1 --run_name smoke_test
  # unsanitized baseline (paper: 0.9251 accuracy)
  python finetune_sanitized.py --data_dir data/SST-2 --run_name SST-2_unsanitized --fp16
  # sanitized data written by run_SanText.py
  python finetune_sanitized.py --data_dir output_SanText_glove/SST-2/eps_3.00 --fp16
  # long texts (e.g. Dreaddit): longer inputs, same effective batch of 64 in two halves
  python finetune_sanitized.py --data_dir new_data/dreaddit --max_seq_length 256 --grad_accum 2 --fp16
"""
import argparse
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from torch.utils.data import DataLoader

import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def parse_args():
    p = argparse.ArgumentParser(description="Fine-tune BERT on (sanitized) TSV data and report dev-set utility.")
    p.add_argument("--data_dir", required=True, help="folder containing train.tsv and dev.tsv")
    p.add_argument("--run_name", default=None, help="results folder name (default: built from data_dir)")
    p.add_argument("--output_root", default="results")
    p.add_argument("--model_name", default="bert-base-uncased")
    p.add_argument("--max_seq_length", type=int, default=128)
    p.add_argument("--batch_size", type=int, default=64, help="effective batch size per optimizer step")
    p.add_argument("--grad_accum", type=int, default=1,
                   help="split each batch into this many micro-batches to save GPU memory (effective batch unchanged)")
    p.add_argument("--eval_batch_size", type=int, default=128)
    p.add_argument("--learning_rate", type=float, default=2e-5)
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--warmup_ratio", type=float, default=0.0)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--max_grad_norm", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--fp16", action="store_true", help="mixed precision on GPU (faster on T4/P100-class GPUs)")
    p.add_argument("--format", choices=["auto", "single", "pair"], default="auto")
    p.add_argument("--max_train_samples", type=int, default=None, help="use only the first N training rows (smoke tests)")
    p.add_argument("--log_every", type=int, default=200, help="print training loss every N optimizer steps")
    p.add_argument("--save_model", action="store_true", help="also save the fine-tuned model (about 440 MB)")
    args = p.parse_args()
    if args.batch_size % args.grad_accum != 0:
        p.error("--batch_size must be divisible by --grad_accum")
    return args


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_tsv(path, fmt):
    """Return (format, texts_a, texts_b or None, labels, skipped_rows)."""
    texts_a, texts_b, labels, skipped = [], [], [], 0
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        if fmt == "auto":
            fmt = "single" if len(header) == 2 else "pair"
        for line in f:
            cols = line.rstrip("\n").split("\t")
            if fmt == "single":
                text, label = "\t".join(cols[:-1]).strip(), cols[-1].strip()
                if len(cols) < 2 or not text or not label:
                    skipped += 1
                    continue
                texts_a.append(text)
            else:
                if len(cols) < 4 or not cols[-1].strip():
                    skipped += 1
                    continue
                texts_a.append(cols[1])
                texts_b.append(cols[2])
                label = cols[-1].strip()
            labels.append(label)
    return fmt, texts_a, (texts_b if fmt == "pair" else None), labels, skipped


def build_label_map(labels):
    def key(x):
        return (0, int(x), "") if x.lstrip("-").isdigit() else (1, 0, x)
    return {lab: i for i, lab in enumerate(sorted(set(labels), key=key))}


def make_examples(texts_a, texts_b, labels, label_map, split_name):
    unknown = sorted(set(labels) - set(label_map))
    if unknown:
        raise ValueError(f"{split_name}.tsv has labels not seen in train.tsv: {unknown[:5]}")
    b = texts_b if texts_b is not None else [None] * len(texts_a)
    return [(a, bb, label_map[lab]) for a, bb, lab in zip(texts_a, b, labels)]


def make_collate(tokenizer, max_len):
    def collate(batch):
        a = [ex[0] for ex in batch]
        b = [ex[1] for ex in batch] if batch[0][1] is not None else None
        enc = tokenizer(a, b, truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([ex[2] for ex in batch], dtype=torch.long)
        return enc
    return collate


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):  # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


def amp(enabled):
    """Mixed-precision context on GPU; a no-op otherwise."""
    return torch.autocast(device_type="cuda", dtype=torch.float16) if enabled else nullcontext()


def compute_metrics(gold, pred, n_labels):
    gold, pred = np.asarray(gold), np.asarray(pred)
    f1s = []
    for c in range(n_labels):
        tp = int(((pred == c) & (gold == c)).sum())
        fp = int(((pred == c) & (gold != c)).sum())
        fn = int(((pred != c) & (gold == c)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    out = {"accuracy": float((gold == pred).mean()), "macro_f1": float(np.mean(f1s))}
    if n_labels == 2:
        out["f1_label_index_1"] = float(f1s[1])  # positive class for SST-2, "stress" for Dreaddit
    return out


@torch.no_grad()
def evaluate(model, loader, device, use_fp16, n_labels):
    model.eval()
    preds, golds = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with amp(use_fp16):
            logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
        preds.extend(logits.argmax(dim=-1).tolist())
        golds.extend(batch["labels"].tolist())
    model.train()
    return compute_metrics(golds, preds, n_labels), preds


def main():
    args = parse_args()
    set_seed(args.seed)
    t_start = time.time()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_fp16 = args.fp16 and device.type == "cuda"
    if args.fp16 and not use_fp16:
        print("note: --fp16 ignored because no GPU is available")
    gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else None

    # ---- data -------------------------------------------------------------
    fmt, tr_a, tr_b, tr_y, tr_skip = read_tsv(os.path.join(args.data_dir, "train.tsv"), args.format)
    _, dv_a, dv_b, dv_y, dv_skip = read_tsv(os.path.join(args.data_dir, "dev.tsv"), fmt)
    if args.max_train_samples:
        tr_a, tr_y = tr_a[: args.max_train_samples], tr_y[: args.max_train_samples]
        tr_b = tr_b[: args.max_train_samples] if tr_b is not None else None
    label_map = build_label_map(tr_y)
    n_labels = len(label_map)
    train_ex = make_examples(tr_a, tr_b, tr_y, label_map, "train")
    dev_ex = make_examples(dv_a, dv_b, dv_y, label_map, "dev")
    print(f"data: {args.data_dir} | format={fmt} | train={len(train_ex)} dev={len(dev_ex)} "
          f"| skipped rows train={tr_skip} dev={dv_skip} | labels={label_map}")

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    collate = make_collate(tokenizer, args.max_seq_length)
    micro_bs = args.batch_size // args.grad_accum
    gen = torch.Generator().manual_seed(args.seed)
    train_loader = DataLoader(train_ex, batch_size=micro_bs, shuffle=True, collate_fn=collate, generator=gen)
    dev_loader = DataLoader(dev_ex, batch_size=args.eval_batch_size, shuffle=False, collate_fn=collate)

    # ---- model, optimizer, schedule ------------------------------------------
    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=n_labels).to(device)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(groups, lr=args.learning_rate, eps=1e-8)
    steps_per_epoch = math.ceil(len(train_loader) / args.grad_accum)
    total_steps = steps_per_epoch * args.epochs
    warmup_steps = int(args.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = make_scaler(use_fp16)

    # ---- training -------------------------------------------------------------
    print(f"training on {gpu_name or 'CPU'} | {args.epochs} epochs x {steps_per_epoch} steps | "
          f"batch {args.batch_size} (micro-batch {micro_bs}) | lr {args.learning_rate} | fp16={use_fp16}")
    history, global_step = [], 0
    model.train()
    for epoch in range(1, args.epochs + 1):
        running_loss, n_loss = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for i, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with amp(use_fp16):
                loss = model(**batch).loss / args.grad_accum
            scaler.scale(loss).backward()
            running_loss += loss.item() * args.grad_accum
            n_loss += 1
            if i % args.grad_accum == 0 or i == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if global_step % args.log_every == 0:
                    print(f"  epoch {epoch} step {global_step}/{total_steps} | loss {running_loss / n_loss:.4f} "
                          f"| lr {scheduler.get_last_lr()[0]:.2e} | {(time.time() - t_start) / 60:.1f} min")
        dev_metrics, preds = evaluate(model, dev_loader, device, use_fp16, n_labels)
        history.append({"epoch": epoch, "train_loss": running_loss / max(1, n_loss), "dev": dev_metrics,
                        "elapsed_min": round((time.time() - t_start) / 60, 2)})
        print(f"epoch {epoch} done | train loss {running_loss / max(1, n_loss):.4f} | dev {dev_metrics}")

    # ---- results (final = after the last epoch, as the original run_glue.py reports) ----
    final_metrics = history[-1]["dev"]
    run_name = args.run_name or "_".join(
        [p for p in os.path.normpath(os.path.abspath(args.data_dir)).split(os.sep) if p][-3:])
    out_dir = os.path.join(args.output_root, run_name)
    os.makedirs(out_dir, exist_ok=True)
    id2label = {v: k for k, v in label_map.items()}
    with open(os.path.join(out_dir, "dev_predictions.tsv"), "w", encoding="utf-8") as f:
        f.write("text_a\ttext_b\tgold\tpred\n")
        for (a, b, gold), pred in zip(dev_ex, preds):
            f.write(f"{a}\t{b or ''}\t{id2label[gold]}\t{id2label[pred]}\n")
    runtime_min = round((time.time() - t_start) / 60, 2)
    report = {
        "run_name": run_name,
        "data_dir": os.path.abspath(args.data_dir),
        "format": fmt,
        "label_map": label_map,
        "n_train": len(train_ex),
        "n_dev": len(dev_ex),
        "skipped_rows": {"train": tr_skip, "dev": dv_skip},
        "settings": vars(args),
        "environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "device": gpu_name or "CPU",
            "platform": platform.platform(),
        },
        "epochs": history,
        "final_dev": final_metrics,
        "runtime_min": runtime_min,
    }
    with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    if args.save_model:
        model.save_pretrained(os.path.join(out_dir, "model"))
        tokenizer.save_pretrained(os.path.join(out_dir, "model"))
    summary = " ".join(f"{k}={v:.4f}" for k, v in final_metrics.items())
    print(f"RESULT {run_name} | {summary} | n_dev={len(dev_ex)} | {runtime_min} min on {gpu_name or 'CPU'}")
    print(f"saved to {out_dir}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile measure.py
"""Run a command, then print its wall time and peak memory (resident set size)."""
import resource
import subprocess
import sys
import time

t0 = time.time()
code = subprocess.run(sys.argv[1:]).returncode
peak_gb = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss * 1024 / 1e9  # ru_maxrss is in KiB on Linux
print(f"[measure] exit={code} wall={time.time() - t0:.0f}s peak_memory={peak_gb:.2f} GB", flush=True)
sys.exit(code)

In [ ]:
%%writefile filter_glove.py
#!/usr/bin/env python3
"""
filter_glove.py: write the part of GloVe that run_SanText.py can use for the given datasets.

run_SanText.py (Yue et al., 2021) reads all ~2.2 million lines of glove.840B.300d.txt on every
run. It keeps a line only if the line's word, after NFD normalisation, is in the dataset's
vocabulary, and the first matching line wins. This script copies all those candidate lines,
unchanged and in their original order, to a small file. Given the small file, run_SanText.py
therefore selects the same words, the same vectors and the same word order as from the full
file, so its output is unchanged; each run just loads in seconds instead of minutes.

It also keeps lines whose word matches a vocabulary word only after both are NFD-normalised
(e.g. a precomposed "café" in the data). run_SanText.py never selects these, because it compares
the normalised GloVe word with the raw token, so they do not change its output; they let the
analysis count the words that miss their vector for this reason.

The vocabulary is built with the authors' own get_vocab_SST2 and spaCy's blank English
tokenizer, exactly as run_SanText.py builds it for --task SST-2 (the single-sentence format
that SST-2, Dreaddit and the app-review dataset all use).

Usage
  python filter_glove.py --glove data/glove.840B.300d.txt --out data/glove.filtered.txt \
      --santext_dir SanText DATA_DIR [DATA_DIR ...]
"""
import argparse
import json
import os
import sys
import time
import unicodedata


def main():
    ap = argparse.ArgumentParser(description="Keep only the GloVe lines run_SanText.py can use for these datasets.")
    ap.add_argument("--glove", required=True, help="full GloVe text file")
    ap.add_argument("--out", required=True, help="filtered GloVe file to write")
    ap.add_argument("--santext_dir", required=True, help="the authors' repository (provides utils.get_vocab_SST2)")
    ap.add_argument("data_dirs", nargs="+", help="folders holding train.tsv and dev.tsv in SST-2 format")
    args = ap.parse_args()

    sys.path.insert(0, os.path.abspath(args.santext_dir))
    from utils import get_vocab_SST2  # the authors' function, unchanged
    from spacy.lang.en import English

    tokenizer = English()
    vocab, sizes = set(), {}
    for d in args.data_dirs:
        v = get_vocab_SST2(d, tokenizer, tokenizer_type="word")
        sizes[d] = len(v)
        vocab.update(v)
        print(f"vocabulary of {d}: {len(v)} word types", flush=True)
    keep = vocab | {unicodedata.normalize("NFD", t) for t in vocab}

    t0 = time.time()
    n_lines = n_kept = 0
    tmp = args.out + ".part"
    # Same reading as run_SanText.py: default text encoding, words split on single spaces.
    with open(args.glove) as f, open(tmp, "w") as out:
        for row in f:
            n_lines += 1
            word = row.rstrip().split(" ")[0]
            if unicodedata.normalize("NFD", word) in keep:
                out.write(row)
                n_kept += 1
    os.replace(tmp, args.out)

    info = {"glove": os.path.abspath(args.glove), "lines_read": n_lines, "lines_kept": n_kept,
            "union_vocabulary": len(vocab), "vocabulary_per_dataset": sizes,
            "seconds": round(time.time() - t0, 1)}
    with open(args.out + ".json", "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    print(f"kept {n_kept:,} of {n_lines:,} GloVe lines for {len(vocab):,} word types -> {args.out} "
          f"({info['seconds']} s)", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sanitize_queue.py
#!/usr/bin/env python3
"""
sanitize_queue.py: run the planned SanText sanitizations one after another, in the background.

The notebook writes plan.json and starts this script, then fine-tunes on the GPU while this
script keeps the CPU busy. Every job calls the authors' unchanged run_SanText.py (through
measure.py, which records wall time and peak memory). The queue:
  1. waits for the full GloVe download, then builds the filtered GloVe file (filter_glove.py),
     which gives run_SanText.py the same words and vectors but loads in seconds;
  2. runs each job whose tier is within config.max_tier, in plan order, skipping jobs that a
     previous session already finished;
  3. marks each output folder with .done (or .failed), so the fine-tuning loop knows when its
     input is ready, and copies the output and its log to the Drive backup folder straight away.

Usage: python sanitize_queue.py plan.json
"""
import json
import os
import shutil
import subprocess
import sys
import time


def backup(path, cfg):
    root = cfg.get("backup_root")
    if not root or not os.path.exists(path):
        return
    dest = os.path.join(root, os.path.relpath(path, cfg["base"]))
    if os.path.isdir(path):
        shutil.copytree(path, dest, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(path, dest)


def write_status(cfg, status):
    status["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    tmp = cfg["status"] + ".part"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(status, f, indent=2)
    os.replace(tmp, cfg["status"])


def last_measure_line(log_path):
    """The '[measure] exit=.. wall=..s peak_memory=.. GB' line that measure.py prints last."""
    try:
        with open(log_path, errors="ignore") as f:
            lines = [l.strip() for l in f if l.startswith("[measure]")]
        return lines[-1] if lines else None
    except OSError:
        return None


def build_filtered_glove(cfg, status):
    if os.path.exists(cfg["glove_filtered"]):
        try:
            with open(cfg["glove_filtered"] + ".json", encoding="utf-8") as f:
                covered = set(json.load(f)["vocabulary_per_dataset"])
        except (OSError, ValueError, KeyError):
            covered = set()
        if set(cfg["filter_data_dirs"]) <= covered:
            print("filtered GloVe already present:", cfg["glove_filtered"], flush=True)
            return True
        print("filtered GloVe misses a dataset; rebuilding it", flush=True)
        os.remove(cfg["glove_filtered"])
    t0 = time.time()
    while not os.path.exists(cfg["glove_ready"]):
        status["current"] = f"waiting for the GloVe download ({(time.time() - t0) / 60:.0f} min)"
        write_status(cfg, status)
        if time.time() - t0 > cfg.get("glove_timeout_min", 90) * 60:
            print("GloVe download did not finish; see logs/glove_download.log", flush=True)
            return False
        time.sleep(15)
    status["current"] = "filtering GloVe"
    write_status(cfg, status)
    log = os.path.join(cfg["logs"], "filter_glove.log")
    cmd = [sys.executable, "-u", cfg["filter"], "--glove", cfg["glove_full"], "--out", cfg["glove_filtered"],
           "--santext_dir", cfg["santext_dir"], *cfg["filter_data_dirs"]]
    with open(log, "w") as lf:
        code = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT).returncode
    print(f"filter_glove.py exit={code} ({time.time() - t0:.0f} s); log: {log}", flush=True)
    if code == 0:
        backup(cfg["glove_filtered"], cfg)
        backup(cfg["glove_filtered"] + ".json", cfg)
        backup(log, cfg)
    return code == 0


def run_job(job, cfg):
    out = job["out_path"]
    log = os.path.join(cfg["logs"], f"san__{job['id']}.log")
    cmd = [sys.executable, "-u", cfg["measure"], sys.executable, "-u", "run_SanText.py",
           "--task", "SST-2", "--method", job["method"], "--epsilon", str(job["epsilon"]),
           "--seed", str(job["seed"]), "--word_embedding_path", cfg["glove_filtered"],
           "--word_embedding_size", "300", "--data_dir", job["data_dir"] + "/",
           "--output_dir", job["output_dir"] + "/", "--threads", str(job["threads"])]
    if job["method"] == "SanText_plus":
        cmd += ["--p", str(job["p"]), "--sensitive_word_percentage", str(job["w"])]
    t0 = time.time()
    with open(log, "w") as lf:
        lf.write("command: " + " ".join(cmd) + "\n")
        lf.flush()
        code = subprocess.run(cmd, cwd=cfg["santext_dir"], stdout=lf, stderr=subprocess.STDOUT).returncode
    ok = code == 0 and all(os.path.exists(os.path.join(out, f)) for f in ("train.tsv", "dev.tsv"))
    os.makedirs(out, exist_ok=True)
    marker = {"id": job["id"], "exit": code, "seconds": round(time.time() - t0), "measure": last_measure_line(log),
              "log": os.path.relpath(log, cfg["base"])}
    with open(os.path.join(out, ".done" if ok else ".failed"), "w", encoding="utf-8") as f:
        json.dump(marker, f)
    backup(out, cfg)
    backup(log, cfg)
    return ok, marker


def main():
    with open(sys.argv[1], encoding="utf-8") as f:
        plan = json.load(f)
    cfg = plan["config"]
    os.makedirs(cfg["logs"], exist_ok=True)
    status = {"current": None, "done": [], "failed": [], "skipped_already_done": [], "not_in_tier": []}
    write_status(cfg, status)

    if not build_filtered_glove(cfg, status):
        status["current"] = "stopped: no GloVe"
        status["failed"] = [j["id"] for j in plan["sanitize"]]
        for j in plan["sanitize"]:
            os.makedirs(j["out_path"], exist_ok=True)
            with open(os.path.join(j["out_path"], ".failed"), "w") as f:
                f.write('{"reason": "GloVe unavailable"}')
        write_status(cfg, status)
        return

    for job in plan["sanitize"]:
        if job["tier"] > cfg["max_tier"]:
            status["not_in_tier"].append(job["id"])
            continue
        if os.path.exists(os.path.join(job["out_path"], ".done")):
            status["skipped_already_done"].append(job["id"])
            continue
        failed_marker = os.path.join(job["out_path"], ".failed")
        if os.path.exists(failed_marker):
            os.remove(failed_marker)  # a new session retries failed jobs once
        status["current"] = job["id"]
        write_status(cfg, status)
        print(f"sanitizing {job['id']} ...", flush=True)
        ok, marker = run_job(job, cfg)
        (status["done"] if ok else status["failed"]).append(job["id"])
        print(f"  {'done' if ok else 'FAILED'}: {job['id']} | {marker['measure']}", flush=True)
        write_status(cfg, status)

    status["current"] = "finished"
    write_status(cfg, status)
    backup(cfg["status"], cfg)
    print("queue finished", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyse_runs.py
#!/usr/bin/env python3
"""
analyse_runs.py: summarise the final runs for the report.

Reads plan.json (written by the notebook) and writes, under --out:
  runs.csv           one row per fine-tuning run: dataset, method, epsilon, seed, accuracy, macro-F1,
                     F1 of label 1, plus the 9 Oct runs if their metrics were copied in
  survival.csv       one row per sanitized output (dev split): share of tokens left unchanged, overall
                     and by word class (content words, words without a GloVe vector, the rarest 90%
                     that SanText+ treats as sensitive, the frequent 10% it treats as non-sensitive)
  resources.csv      vocabulary, matrix size, wall time and peak memory of every sanitization
  aligned/<id>.tsv   review test split only: original and sanitized token at every position, for
                     the analysis against the human-flagged sensitive words
  vocab/<name>.tsv   the authors' vocabulary for each dataset: frequency rank, count, whether GloVe
                     has a vector, whether SanText+ (w = 0.9) treats it as sensitive, and whether the
                     authors' NFD comparison misses a vector GloVe does have
  repro.json         reproducibility checks (same seed with 1 vs 2 worker processes; one review
                     repeated 1,000 times; 9 Oct output vs today's with the same seed)
  summary.md         all of the above as readable tables, with the paper's numbers alongside

Word classes and alignment follow run_SanText.py: texts are tokenized with spaCy's blank English
tokenizer (the authors' tokenizer), the sanitized text is the replacement tokens joined by single
spaces, so position i of the sanitized text replaces token i of the original.
"""
import argparse
import collections
import csv
import json
import os
import re
import statistics
import sys
import unicodedata

STOP = set("the a an and or of to in on for with my me i you your it is was be are this that at by from as not "
           "no but so our we they their its has have had can just all any get got".split())

# Yue et al. (2021), Table 1, GloVe embeddings, SST-2 dev accuracy
PAPER_SST2 = {("unsanitized", None): 0.9251,
              ("SanText", 1.0): 0.5101, ("SanText", 2.0): 0.5838, ("SanText", 3.0): 0.8374,
              ("SanText_plus", 1.0): 0.7796, ("SanText_plus", 2.0): 0.7943, ("SanText_plus", 3.0): 0.8516}
METHOD_NAME = {"unsanitized": "Unsanitized", "SanText": "SanText", "SanText_plus": "SanText+"}


def read_original_texts(path):
    """As run_SanText.py reads them: strip the line, take the first tab-separated field."""
    with open(path, encoding="utf-8") as f:
        next(f)
        return [line.strip().split("\t")[0] for line in f]


def read_sanitized_texts(path):
    """run_SanText.py writes '<tokens joined by spaces>\\t<label>' after a header row."""
    with open(path, encoding="utf-8") as f:
        next(f)
        return [line.rstrip("\n").rsplit("\t", 1)[0] for line in f]


def load_glove_words(path):
    words = set()
    with open(path) as f:
        for row in f:
            words.add(unicodedata.normalize("NFD", row.rstrip().split(" ")[0]))
    return words


def authors_vocab(data_dir, tokenizer, w=0.9):
    from utils import get_vocab_SST2  # the authors' function
    vocab = get_vocab_SST2(data_dir, tokenizer, tokenizer_type="word")
    words = [k for k, _ in vocab.most_common()]
    n_sensitive = int(w * len(vocab))
    sensitive = set(words[-n_sensitive - 1:])  # the authors' slice, including its extra word
    return vocab, words, sensitive


def is_content(tok):
    return tok.isalpha() and len(tok) >= 3 and tok.lower() not in STOP


def mean_sd(values):
    if not values:
        return None, None
    if len(values) == 1:
        return values[0], None
    return statistics.mean(values), statistics.stdev(values)


def fmt(x, nd=4):
    return "–" if x is None else f"{x:.{nd}f}"


def compare_files(a_dir, b_dir):
    """Share of identical lines (and of identical tokens where lengths match) between two outputs."""
    out = {}
    for split in ("train", "dev"):
        pa, pb = os.path.join(a_dir, f"{split}.tsv"), os.path.join(b_dir, f"{split}.tsv")
        if not (os.path.exists(pa) and os.path.exists(pb)):
            continue
        a, b = read_sanitized_texts(pa), read_sanitized_texts(pb)
        n = min(len(a), len(b))
        same_lines = sum(x == y for x, y in zip(a[:n], b[:n]))
        tok_total = tok_same = 0
        for x, y in zip(a[:n], b[:n]):
            xs, ys = x.split(" "), y.split(" ")
            if len(xs) == len(ys):
                tok_total += len(xs)
                tok_same += sum(p == q for p, q in zip(xs, ys))
        out[split] = {"lines": n, "identical_lines": same_lines, "share_identical_lines": round(same_lines / max(1, n), 4),
                      "share_identical_tokens": round(tok_same / max(1, tok_total), 4)}
    return out


def main():
    ap = argparse.ArgumentParser(description="Summarise the final runs.")
    ap.add_argument("--plan", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()

    with open(args.plan, encoding="utf-8") as f:
        plan = json.load(f)
    cfg = plan["config"]
    sys.path.insert(0, cfg["santext_dir"])
    from spacy.lang.en import English
    tokenizer = English()
    os.makedirs(os.path.join(args.out, "aligned"), exist_ok=True)
    os.makedirs(os.path.join(args.out, "vocab"), exist_ok=True)
    md = ["# Final runs: summary", ""]

    env_path = os.path.join(args.out, "environment.json")
    if os.path.exists(env_path):
        with open(env_path, encoding="utf-8") as f:
            env = json.load(f)
        md += ["Environment: " + ", ".join(f"{k} {v}" for k, v in env.items()), ""]

    # ---------------------------------------------------------------- vocabularies
    glove = load_glove_words(cfg["glove_filtered"]) if os.path.exists(cfg["glove_filtered"]) else None
    vinfo, vsummary = {}, []
    for name, data_dir in plan["datasets"].items():
        if not os.path.exists(os.path.join(data_dir, "dev.tsv")):
            continue
        vocab, words, sensitive = authors_vocab(data_dir, tokenizer)
        has_vec = {t: (glove is not None and t in glove) for t in words}
        nfd_miss = {t: (glove is not None and t != unicodedata.normalize("NFD", t)
                        and unicodedata.normalize("NFD", t) in glove) for t in words}
        vinfo[name] = {"vocab": vocab, "sensitive": sensitive, "has_vec": has_vec}
        with open(os.path.join(args.out, "vocab", f"{name}.tsv"), "w", encoding="utf-8", newline="") as f:
            w = csv.writer(f, delimiter="\t")
            w.writerow(["rank", "token", "count", "has_glove_vector", "sensitive_w0.9", "nfd_mismatch"])
            for r, t in enumerate(words, start=1):
                w.writerow([r, t, vocab[t], int(has_vec[t]), int(t in sensitive), int(nfd_miss[t])])
        vsummary.append((name, len(words), sum(has_vec.values()), len(words) - sum(has_vec.values()),
                         sum(nfd_miss.values()), len(sensitive), sum(1 for t in sensitive if has_vec[t])))
    if vsummary:
        md += ["## Vocabularies (the authors' construction: train + dev, spaCy blank English tokenizer)", "",
               "| Dataset | Word types | With GloVe vector | Without (replaced uniformly at random) | of which NFD mismatch | Sensitive, w = 0.9 | Sensitive with vector |",
               "|---|---|---|---|---|---|---|"]
        md += [f"| {n} | {a:,} | {b:,} | {c:,} | {d:,} | {e:,} | {g:,} |" for n, a, b, c, d, e, g in vsummary]
        md.append("")

    # ---------------------------------------------------------------- sanitizations: resources + survival
    jobs = {j["id"]: j for j in plan["sanitize"]}
    res_rows, surv_rows = [], []
    for jid, job in jobs.items():
        out = job["out_path"]
        if not os.path.exists(os.path.join(out, ".done")):
            continue
        log_path = os.path.join(cfg["logs"], f"san__{jid}.log")
        log = open(log_path, errors="ignore").read() if os.path.exists(log_path) else ""
        tw = re.search(r"#Total Words: (\d+), #Sensitive Words: (\d+)", log)
        mat = re.search(r"All Word Embedding Matrix: \((\d+), (\d+)\)", log)
        smat = re.search(r"Sensitive Word Embedding Matrix: \((\d+), (\d+)\)", log)
        meas = re.search(r"\[measure\] exit=(\d+) wall=(\d+)s peak_memory=([\d.]+) GB", log)
        res_rows.append({"job": jid, "dataset": job["dataset"], "method": job["method"], "epsilon": job["epsilon"],
                         "seed": job["seed"], "threads": job["threads"],
                         "total_words": tw.group(1) if tw else "", "sensitive_words": tw.group(2) if tw else "",
                         "rows_with_vector": mat.group(1) if mat else "", "columns_sensitive_with_vector": smat.group(1) if smat else "",
                         "wall_s": meas.group(2) if meas else "", "peak_gb": meas.group(3) if meas else ""})

        info = vinfo.get(job["dataset"])
        if info is None or job.get("diagnostic"):
            continue
        orig = read_original_texts(os.path.join(job["data_dir"], "dev.tsv"))
        san = read_sanitized_texts(os.path.join(out, "dev.tsv"))
        c = collections.Counter()
        aligned_rows = []
        for d, (o_text, s_text) in enumerate(zip(orig, san)):
            o = [t.text for t in tokenizer(o_text)]
            s = s_text.split(" ") if s_text else []
            c["docs"] += 1
            if len(o) != len(s):
                continue
            c["docs_aligned"] += 1
            for pos, (ot, st) in enumerate(zip(o, s)):
                same = ot == st
                classes = ["all"]
                if is_content(ot):
                    classes.append("content")
                if not info["has_vec"].get(ot, False):
                    classes.append("no_vector")
                elif ot in info["sensitive"]:
                    classes.append("rare90")
                else:
                    classes.append("frequent10")
                for k in classes:
                    c[f"n_{k}"] += 1
                    c[f"kept_{k}"] += same
                if job["dataset"] == "reviews":
                    aligned_rows.append((d, pos, ot, st))
        row = {"job": jid, "dataset": job["dataset"], "method": job["method"], "epsilon": job["epsilon"],
               "seed": job["seed"], "threads": job["threads"], "docs": c["docs"], "docs_aligned": c["docs_aligned"]}
        for k in ("all", "content", "rare90", "frequent10", "no_vector"):
            row[f"tokens_{k}"] = c[f"n_{k}"]
            row[f"unchanged_{k}"] = c[f"kept_{k}"]
            row[f"share_{k}"] = round(c[f"kept_{k}"] / c[f"n_{k}"], 4) if c[f"n_{k}"] else ""
        surv_rows.append(row)
        if aligned_rows:
            with open(os.path.join(args.out, "aligned", f"{jid}.tsv"), "w", encoding="utf-8", newline="") as f:
                w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
                w.writerow(["doc", "pos", "orig", "san"])
                w.writerows(aligned_rows)

    for name, rows in (("resources.csv", res_rows), ("survival.csv", surv_rows)):
        if rows:
            with open(os.path.join(args.out, name), "w", encoding="utf-8", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(rows[0]))
                w.writeheader()
                w.writerows(rows)

    if surv_rows:
        md += ["## Tokens left unchanged by sanitization (dev split)", "",
               "rare 90% = the words SanText+ treats as sensitive (w = 0.9); frequent 10% = the rest; "
               "no vector = words GloVe lacks, replaced uniformly at random.", "",
               "| Dataset | Method | ε | Seed | Docs aligned | All | Content words | Rare 90% | Frequent 10% | No vector |",
               "|---|---|---|---|---|---|---|---|---|---|"]
        for r in sorted(surv_rows, key=lambda r: (r["dataset"], r["method"], r["epsilon"], r["seed"])):
            pct = lambda k: "–" if r[f"share_{k}"] == "" else f"{100 * r[f'share_{k}']:.1f}%"
            md.append(f"| {r['dataset']} | {METHOD_NAME[r['method']]} | {r['epsilon']:g} | {r['seed']} | "
                      f"{r['docs_aligned']}/{r['docs']} | {pct('all')} | {pct('content')} | {pct('rare90')} | "
                      f"{pct('frequent10')} | {pct('no_vector')} |")
        md.append("")
    if res_rows:
        md += ["## Sanitization resources", "",
               "| Job | Words | Rows with vector | Sensitive columns | Wall (s) | Peak memory (GB) |", "|---|---|---|---|---|---|"]
        md += [f"| {r['job']} | {r['total_words']} | {r['rows_with_vector']} | {r['columns_sensitive_with_vector']} | "
               f"{r['wall_s']} | {r['peak_gb']} |" for r in res_rows]
        md.append("")

    # ---------------------------------------------------------------- fine-tuning results
    runs = []
    for job in plan["finetune"]:
        p = os.path.join(cfg["base"], "results", job["run_name"], "metrics.json")
        if not os.path.exists(p):
            continue
        with open(p, encoding="utf-8") as f:
            m = json.load(f)
        d = m["final_dev"]
        runs.append({"run": job["run_name"], "source": "this notebook", "dataset": job["dataset"], "method": job["method"],
                     "epsilon": job["epsilon"], "seed": job["seed"], "n_train": m["n_train"], "n_dev": m["n_dev"],
                     "accuracy": d["accuracy"], "macro_f1": d["macro_f1"], "f1_label1": d.get("f1_label_index_1"),
                     "minutes": m["runtime_min"], "transformers": m["environment"]["transformers"],
                     "torch": m["environment"]["torch"]})
    for old in plan.get("old_runs", []):
        p = os.path.join(old["path"], "metrics.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                m = json.load(f)
            d = m["final_dev"]
            runs.append({"run": old["run_name"], "source": "9 Oct", "dataset": old["dataset"], "method": old["method"],
                         "epsilon": old["epsilon"], "seed": old["seed"], "n_train": m["n_train"], "n_dev": m["n_dev"],
                         "accuracy": d["accuracy"], "macro_f1": d["macro_f1"], "f1_label1": d.get("f1_label_index_1"),
                         "minutes": m["runtime_min"], "transformers": m["environment"]["transformers"],
                         "torch": m["environment"]["torch"]})
    if runs:
        with open(os.path.join(args.out, "runs.csv"), "w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(runs[0]))
            w.writeheader()
            w.writerows(runs)

        cur = [r for r in runs if r["source"] == "this notebook"]
        sst = [r for r in cur if r["dataset"] == "sst2"]
        if sst:
            md += ["## SST-2 dev accuracy against the paper (Yue et al., 2021, Table 1, GloVe)", "",
                   "| Method | ε | Paper | Ours, per seed | Mean | SD | Mean − paper |", "|---|---|---|---|---|---|---|"]
            for (method, eps), paper in PAPER_SST2.items():
                vals = sorted((r["seed"], r["accuracy"]) for r in sst if r["method"] == method and r["epsilon"] == eps)
                if not vals:
                    continue
                mu, sd = mean_sd([v for _, v in vals])
                per_seed = ", ".join(f"{v:.4f} (s{s})" for s, v in vals)
                eps_txt = "–" if eps is None else f"{eps:g}"
                md.append(f"| {METHOD_NAME[method]} | {eps_txt} | {paper:.4f} | {per_seed} | "
                          f"{fmt(mu)} | {fmt(sd)} | {mu - paper:+.4f} |")
            md.append("")
        new = [r for r in cur if r["dataset"] != "sst2"]
        if new:
            md += ["## New datasets (dev = held-out test split)", "",
                   "| Dataset | Method | ε | Seed | Accuracy | Macro-F1 | F1 (label 1) |", "|---|---|---|---|---|---|---|"]
            for r in sorted(new, key=lambda r: (r["dataset"], r["method"] != "unsanitized", r["method"], r["epsilon"] or 0, r["seed"])):
                eps = "–" if r["epsilon"] is None else f"{r['epsilon']:g}"
                md.append(f"| {r['dataset']} | {METHOD_NAME[r['method']]} | {eps} | "
                          f"{r['seed']} | {r['accuracy']:.4f} | {r['macro_f1']:.4f} | {fmt(r['f1_label1'])} |")
            md.append("")
        old = [r for r in runs if r["source"] == "9 Oct"]
        if old:
            md += ["## 9 Oct runs, for comparison", "", "| Run | Accuracy | Macro-F1 | transformers |", "|---|---|---|---|"]
            md += [f"| {r['run']} | {r['accuracy']:.4f} | {r['macro_f1']:.4f} | {r['transformers']} |" for r in old]
            md.append("")

    # ---------------------------------------------------------------- reproducibility
    repro = {}
    for pair in plan.get("repro", {}).get("pairs", []):
        a, b = jobs[pair["a"]], jobs[pair["b"]]
        if all(os.path.exists(os.path.join(j["out_path"], ".done")) for j in (a, b)):
            repro[pair["name"]] = {"description": pair["description"], **compare_files(a["out_path"], b["out_path"])}
    for item in plan.get("repro", {}).get("dupdoc", []):
        j = jobs[item["job"]]
        if os.path.exists(os.path.join(j["out_path"], ".done")):
            dev = read_sanitized_texts(os.path.join(j["out_path"], "dev.tsv"))
            cnt = collections.Counter(dev)
            repro[item["name"]] = {"description": item["description"], "copies": len(dev), "distinct_outputs": len(cnt),
                                   "copies_sharing_their_output": sum(n for n in cnt.values() if n > 1),
                                   "largest_group": max(cnt.values()) if cnt else 0}
    for item in plan.get("repro", {}).get("old_vs_new", []):
        j = jobs.get(item["job"])
        if j and os.path.isdir(item["old_dir"]) and os.path.exists(os.path.join(j["out_path"], ".done")):
            repro[item["name"]] = {"description": item["description"], **compare_files(item["old_dir"], j["out_path"])}
    if repro:
        with open(os.path.join(args.out, "repro.json"), "w", encoding="utf-8") as f:
            json.dump(repro, f, indent=2)
        md += ["## Reproducibility checks", ""]
        for name, r in repro.items():
            md.append(f"- **{name}**: {r['description']}")
            for k, v in r.items():
                if k != "description":
                    md.append(f"  - {k}: {v}")
        md.append("")

    with open(os.path.join(args.out, "summary.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    print("\n".join(md))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_plan.py
#!/usr/bin/env python3
"""
make_plan.py: the run plan for the final report, in priority order.

Tier 1 (essentials):  new datasets at eps = 1, 2, 3 with SanText and SanText+; SanText+ on SST-2;
                      reproducibility checks (cheap: they use the small review dataset).
Tier 2 (should):      SST-2 SanText at eps = 1, 2, 3 and the unsanitized baseline with seeds 42-44,
                      so the replication reports run-to-run spread instead of single runs.
Tier 3 (could):       SanText+ on SST-2 with seeds 43-44; new datasets at eps = 3 with seeds 43-44.

Every sanitization uses the authors' run_SanText.py with --threads 8 (as on 9 Oct; Colab has two
cores, so this means two worker processes), except where a reproducibility check sets it.
SanText+ uses the paper's p = 0.3 and w = 0.9 (Yue et al., 2021, Sec. 5.1).
Fine-tuning settings match 9 Oct: SST-2 with the paper's settings; Dreaddit with 256-token inputs
and batch 16; reviews with batch 16 (small datasets, so more optimizer steps).
"""
import json
import os

P, W = 0.3, 0.9
EPS = (1.0, 2.0, 3.0)
FT_ARGS = {"sst2": ["--fp16", "--log_every", "300"],
           "dreaddit": ["--max_seq_length", "256", "--batch_size", "16", "--fp16"],
           "reviews": ["--batch_size", "16", "--fp16"]}


def out_path(output_dir, method, eps):
    path = os.path.join(output_dir, "eps_%.2f" % eps)
    return os.path.join(path, "sword_%.2f_p_%.2f" % (W, P)) if method == "SanText_plus" else path


def build_plan(base, santext_dir, data_dirs, max_tier=3, backup_root=None, old_root=None, seeds=(42, 43, 44)):
    """data_dirs: {"sst2": ..., "dreaddit": ..., "reviews": ..., "dupdoc": ...}; missing datasets are left out."""
    have = {k: v for k, v in data_dirs.items() if v and os.path.exists(os.path.join(v, "dev.tsv"))}
    san, ft = [], []

    def S(ds, method, eps, seed, tier, threads=8, tag="", diagnostic=False):
        if ds not in have:
            return None
        jid = f"{ds}__{method}__eps{eps:.2f}__seed{seed}{tag}"
        output_dir = os.path.join(base, "san", ds, method, f"seed_{seed}{tag}")
        job = {"id": jid, "tier": tier, "dataset": ds, "method": method, "epsilon": eps, "seed": seed,
               "threads": threads, "p": P, "w": W, "data_dir": have[ds], "output_dir": output_dir,
               "out_path": out_path(output_dir, method, eps), "diagnostic": diagnostic}
        san.append(job)
        return job

    def F(ds, method, eps, seed, tier, needs=None):
        if ds not in have or (method != "unsanitized" and needs is None):
            return
        name = f"{ds}__unsanitized__seed{seed}" if method == "unsanitized" else f"{ds}__{method}__eps{eps:.2f}__seed{seed}"
        ft.append({"run_name": name, "tier": tier, "dataset": ds, "method": method,
                   "epsilon": None if method == "unsanitized" else eps, "seed": seed,
                   "data_dir": have[ds] if method == "unsanitized" else needs["out_path"],
                   "needs": None if needs is None else needs["id"], "args": FT_ARGS[ds] + ["--seed", str(seed)]})

    # ---------------- sanitizations, in the order the background queue runs them
    s42 = seeds[0]
    new_jobs, repro_jobs = {}, {}
    for ds in ("reviews", "dreaddit"):
        for method in ("SanText", "SanText_plus"):
            for e in EPS:
                new_jobs[(ds, method, e, s42)] = S(ds, method, e, s42, tier=1)
        if ds == "reviews":   # reproducibility checks right after the first (fast) review jobs
            repro_jobs = {
                "t1a": S("reviews", "SanText", 1.0, s42, 1, threads=1, tag="_rep_t1a", diagnostic=True),
                "t1b": S("reviews", "SanText", 1.0, s42, 1, threads=1, tag="_rep_t1b", diagnostic=True),
                "t2a": S("reviews", "SanText", 1.0, s42, 1, threads=8, tag="_rep_t2a", diagnostic=True),
                "t2b": S("reviews", "SanText", 1.0, s42, 1, threads=8, tag="_rep_t2b", diagnostic=True),
                "p1a": S("reviews", "SanText_plus", 1.0, s42, 1, threads=1, tag="_rep_p1a", diagnostic=True),
                "p1b": S("reviews", "SanText_plus", 1.0, s42, 1, threads=1, tag="_rep_p1b", diagnostic=True),
                "dup1": S("dupdoc", "SanText", 1.0, s42, 1, threads=1, tag="_rep_t1", diagnostic=True),
                "dup2": S("dupdoc", "SanText", 1.0, s42, 1, threads=8, tag="_rep_t2", diagnostic=True),
            }
    sst_plus = {(e, s42): S("sst2", "SanText_plus", e, s42, tier=1) for e in EPS}
    sst_san = {(e, s): S("sst2", "SanText", e, s, tier=2) for s in seeds for e in EPS}
    for s in seeds[1:]:
        for e in EPS:
            sst_plus[(e, s)] = S("sst2", "SanText_plus", e, s, tier=3)
    for ds in ("reviews", "dreaddit"):
        for method in ("SanText", "SanText_plus"):
            for s in seeds[1:]:
                new_jobs[(ds, method, 3.0, s)] = S(ds, method, 3.0, s, tier=3)

    # ---------------- fine-tuning, in the order the GPU loop runs them
    for ds in ("reviews", "dreaddit", "sst2"):
        F(ds, "unsanitized", None, s42, tier=1)
    for ds in ("reviews", "dreaddit"):
        for method in ("SanText", "SanText_plus"):
            for e in EPS:
                F(ds, method, e, s42, 1, new_jobs.get((ds, method, e, s42)))
    for e in EPS:
        F("sst2", "SanText_plus", e, s42, 1, sst_plus.get((e, s42)))
    for e in EPS:
        F("sst2", "SanText", e, s42, 2, sst_san.get((e, s42)))
    for s in seeds[1:]:
        F("sst2", "unsanitized", None, s, tier=2)
        for e in EPS:
            F("sst2", "SanText", e, s, 2, sst_san.get((e, s)))
    for s in seeds[1:]:
        for e in EPS:
            F("sst2", "SanText_plus", e, s, 3, sst_plus.get((e, s)))
    for ds in ("reviews", "dreaddit"):
        for method in ("SanText", "SanText_plus"):
            for s in seeds[1:]:
                F(ds, method, 3.0, s, 3, new_jobs.get((ds, method, 3.0, s)))

    # ---------------- reproducibility comparisons
    repro = {"pairs": [], "dupdoc": [], "old_vs_new": []}
    if "reviews" in have and repro_jobs:
        repro["pairs"] = [
            {"name": "SanText, same seed, 1 worker, two runs", "a": repro_jobs["t1a"]["id"], "b": repro_jobs["t1b"]["id"],
             "description": "Reviews, SanText eps = 1, seed 42, --threads 1, run twice. Identical files mean the seed controls the output."},
            {"name": "SanText, same seed, 2 workers, two runs", "a": repro_jobs["t2a"]["id"], "b": repro_jobs["t2b"]["id"],
             "description": "Reviews, SanText eps = 1, seed 42, --threads 8 (two workers on Colab), run twice."},
            {"name": "SanText+, same seed, 1 worker, two runs", "a": repro_jobs["p1a"]["id"], "b": repro_jobs["p1b"]["id"],
             "description": "Reviews, SanText+ eps = 1, seed 42, --threads 1, run twice. The keep/replace coin uses Python's "
                            "random module, which reseeds itself in every worker process, so the seed cannot fix it."}]
    if "dupdoc" in have and repro_jobs.get("dup1"):
        repro["dupdoc"] = [
            {"name": "One review x 1,000, 1 worker", "job": repro_jobs["dup1"]["id"],
             "description": "Dev file = the first test review repeated 1,000 times; SanText eps = 1, --threads 1. "
                            "Independent sampling gives (almost) 1,000 distinct outputs."},
            {"name": "One review x 1,000, 2 workers", "job": repro_jobs["dup2"]["id"],
             "description": "Same input with two worker processes. Each worker inherits the same NumPy random state, "
                            "so copies processed at the same point by different workers get identical outputs."}]
    if old_root:
        for e in EPS:
            j = sst_san.get((e, s42))
            if j:
                repro["old_vs_new"].append({"name": f"SST-2 SanText eps = {e:g}: 9 Oct vs today (seed 42)", "job": j["id"],
                                            "old_dir": os.path.join(old_root, "SST-2", "eps_%.2f" % e),
                                            "description": "Same code, data and seed (42), --threads 8, on two days."})
    return {"config": {"base": base, "santext_dir": santext_dir,
                       "measure": os.path.join(base, "measure.py"), "filter": os.path.join(base, "filter_glove.py"),
                       "glove_full": os.path.join(base, "data", "glove.840B.300d.txt"),
                       "glove_ready": os.path.join(base, "data", "glove.ready"),
                       "glove_filtered": os.path.join(base, "data", "glove.filtered.txt"),
                       "filter_data_dirs": [v for k, v in have.items()],
                       "logs": os.path.join(base, "logs"), "status": os.path.join(base, "logs", "queue_status.json"),
                       "backup_root": backup_root, "max_tier": max_tier},
            "datasets": {k: v for k, v in have.items() if k != "dupdoc"},
            "sanitize": san, "finetune": ft, "repro": repro, "old_runs": []}


if __name__ == "__main__":
    import sys
    plan = build_plan(*sys.argv[1:3], json.loads(sys.argv[3]))
    print(json.dumps({"sanitize": len(plan["sanitize"]), "finetune": len(plan["finetune"])}))

In [ ]:
# Step 1 · Inputs: SST-2, Dreaddit and your review dataset; GloVe downloads in the background (about 5 minutes)
DATA = f"{BASE}/data"


def rows(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f) - 1


def sha1(path):
    with open(path, "rb") as f:
        return hashlib.sha1(f.read()).hexdigest()[:12]


# SST-2 (GLUE), as on 9 Oct
if not os.path.exists(f"{DATA}/SST-2/train.tsv"):
    !cd {DATA} && wget -q -O SST-2.zip https://dl.fbaipublicfiles.com/glue/data/SST-2.zip && unzip -o -q SST-2.zip && rm SST-2.zip
print(f"SST-2: train {rows(f'{DATA}/SST-2/train.tsv'):,} | dev {rows(f'{DATA}/SST-2/dev.tsv'):,}")

# Your constructed dataset: the Drive copy from 9 Oct is the only copy (Google Play now returns other reviews)
def find_reviews(old):
    """The review dataset's folder: the expected place, else a search of My Drive (e.g. a shortcut or an upload)."""
    if os.path.exists(f"{old}/new_data/app_reviews/dev.tsv"):
        return f"{old}/new_data/app_reviews"
    pats = ["/content/drive/MyDrive/*/new_data/app_reviews/dev.tsv", "/content/drive/MyDrive/*/*/new_data/app_reviews/dev.tsv",
            "/content/drive/MyDrive/app_reviews/dev.tsv", "/content/drive/MyDrive/*/app_reviews/dev.tsv"]
    hits = sorted(h for p in pats for h in glob.glob(p))
    return os.path.dirname(hits[0]) if hits else None


REV_SRC = find_reviews(OLD)
if REV_SRC and REV_SRC.endswith("/new_data/app_reviews"):
    OLD = REV_SRC[: -len("/new_data/app_reviews")]   # the 9 Oct folder, wherever it is
print("review dataset folder:", REV_SRC or "NOT FOUND")
RV, DUP = f"{DATA}/app_reviews", None
os.makedirs(RV, exist_ok=True)
for f in ("train.tsv", "dev.tsv", "stats.json", "collection_log.json"):
    src = f"{REV_SRC}/{f}" if REV_SRC else ""
    if os.path.exists(src) and not os.path.exists(f"{RV}/{f}"):
        shutil.copy2(src, f"{RV}/{f}")
REVIEWS_OK = os.path.exists(f"{RV}/train.tsv") and os.path.exists(f"{RV}/dev.tsv")
if REVIEWS_OK:
    n = (rows(f"{RV}/train.tsv"), rows(f"{RV}/dev.tsv"))
    print(f"reviews: train {n[0]:,} | test {n[1]:,} | sha1 {sha1(f'{RV}/train.tsv')} / {sha1(f'{RV}/dev.tsv')}"
          + ("" if n == (2170, 542) else "   !! expected 2,170 / 542: tell Claude"))
    # Reproducibility input: the first test review repeated 1,000 times (vocabulary from the real training set)
    DUP = f"{DATA}/repro_dupdoc"
    os.makedirs(DUP, exist_ok=True)
    shutil.copy2(f"{RV}/train.tsv", f"{DUP}/train.tsv")
    with open(f"{RV}/dev.tsv", encoding="utf-8") as f:
        header, first = f.readline(), f.readline()
    with open(f"{DUP}/dev.tsv", "w", encoding="utf-8") as f:
        f.write(header + first * 1000)
else:
    print(f"!! Review dataset NOT found in {OLD}/new_data/app_reviews. The review runs will be skipped.\n"
          "   Check that the folder is in your Drive, then tell Claude before going further.")

# Dreaddit (Turcan and McKeown, 2019), converted exactly as on 9 Oct
DD = f"{DATA}/dreaddit"
os.makedirs(DD, exist_ok=True)
DREADDIT_OK = os.path.exists(f"{DD}/train.tsv") and os.path.exists(f"{DD}/dev.tsv")
if not DREADDIT_OK:
    import csv, urllib.request, zipfile
    try:
        if not glob.glob(f"{DD}/**/dreaddit-train.csv", recursive=True):
            urllib.request.urlretrieve("http://www.cs.columbia.edu/~eturcan/data/dreaddit.zip", f"{DD}/dreaddit.zip")
            with zipfile.ZipFile(f"{DD}/dreaddit.zip") as z:
                z.extractall(DD)
        for split, name in (("train", "dreaddit-train.csv"), ("dev", "dreaddit-test.csv")):
            src = glob.glob(f"{DD}/**/{name}", recursive=True)[0]
            with open(src, encoding="utf-8", newline="") as f:
                keep = [r for r in csv.DictReader(f)
                        if r.get("text", "").strip() and r.get("label", "").strip() in ("0", "1")]
            with open(f"{DD}/{split}.tsv", "w", encoding="utf-8") as f:
                f.write("sentence\tlabel\n")
                for r in keep:
                    f.write(" ".join(r["text"].split()) + "\t" + r["label"].strip() + "\n")
        DREADDIT_OK = True
    except Exception as e:
        print("!! Dreaddit download failed; its runs will be skipped:", e)
if DREADDIT_OK:
    print(f"Dreaddit: train {rows(f'{DD}/train.tsv'):,} | test {rows(f'{DD}/dev.tsv'):,}")

# 9 Oct outputs, for the comparison tables (optional)
for e in ("1.00", "2.00", "3.00"):
    src = f"{OLD}/SanText/output_SanText_glove/SST-2/eps_{e}"
    if os.path.isdir(src):
        shutil.copytree(src, f"{DATA}/old_san/SST-2/eps_{e}", dirs_exist_ok=True)
OLD_RUNS = sorted(glob.glob(f"{OLD}/results/*/metrics.json"))
print(f"9 Oct: {len(glob.glob(f'{DATA}/old_san/SST-2/eps_*'))} sanitized SST-2 folders, {len(OLD_RUNS)} fine-tuning runs found")

# GloVe (2.2 GB) in the background, unless an earlier session already left the filtered copy on Drive
glove_proc = None
need = {f"{DATA}/SST-2"} | ({DD} if DREADDIT_OK else set()) | ({RV} if REVIEWS_OK else set()) | ({DUP} if DUP else set())
if os.path.exists(f"{DATA}/glove.filtered.txt"):
    try:
        covered = set(json.load(open(f"{DATA}/glove.filtered.txt.json"))["vocabulary_per_dataset"])
    except Exception:
        covered = set()
    if not need <= covered:  # a dataset was added since the filtered file was made: rebuild it
        print("filtered GloVe lacks", sorted(need - covered), "-> downloading GloVe to rebuild it")
        for f in ("glove.filtered.txt", "glove.filtered.txt.json"):
            for root in (DATA, f"{NEW}/data"):
                if os.path.exists(f"{root}/{f}"):
                    os.remove(f"{root}/{f}")
if os.path.exists(f"{DATA}/glove.filtered.txt"):
    print("filtered GloVe restored from Drive: no download needed")
elif not os.path.exists(f"{DATA}/glove.ready"):
    urls = ["https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip",   # mirror, usually faster
            "https://nlp.stanford.edu/data/glove.840B.300d.zip"]                          # original
    cmd = (" || ".join(f"wget -q -O {DATA}/glove.zip {u}" for u in urls)
           + f" && unzip -o -q {DATA}/glove.zip -d {DATA} && rm -f {DATA}/glove.zip && touch {DATA}/glove.ready")
    glove_proc = subprocess.Popen(["bash", "-c", cmd], stdout=open(f"{BASE}/logs/glove_download.log", "w"),
                                  stderr=subprocess.STDOUT)
    print("GloVe download started in the background (the queue waits for it)")

In [ ]:
# Step 2 · The run plan, and the background sanitization queue
MAX_TIER = 1   # 1 = essentials (about 1.5 h of GPU), 2 = + SST-2 seeds (about 3 h), 3 = + extra seeds (about 4 h)

sys.path.insert(0, BASE)
import importlib, make_plan
importlib.reload(make_plan)
plan = make_plan.build_plan(BASE, f"{BASE}/SanText",
                            {"sst2": f"{DATA}/SST-2", "dreaddit": DD if DREADDIT_OK else None,
                             "reviews": RV if REVIEWS_OK else None, "dupdoc": DUP},
                            max_tier=MAX_TIER, backup_root=NEW, old_root=f"{DATA}/old_san")


def old_run(path):
    """Describe a 9 Oct run folder such as SST-2_SanText_eps_3.00 or dreaddit_unsanitized."""
    name = os.path.basename(os.path.dirname(path))
    m = re.match(r"(SST-2|dreaddit|reviews)_(unsanitized|SanText_eps_([\d.]+))$", name)
    if not m:
        return None
    return {"run_name": name, "path": os.path.dirname(path), "dataset": {"SST-2": "sst2"}.get(m.group(1), m.group(1)),
            "method": "unsanitized" if m.group(2) == "unsanitized" else "SanText",
            "epsilon": float(m.group(3)) if m.group(3) else None, "seed": 42}


plan["old_runs"] = [r for r in map(old_run, OLD_RUNS) if r]
with open(f"{BASE}/plan.json", "w") as f:
    json.dump(plan, f, indent=1)
backup(f"{BASE}/plan.json")

for key, label in (("sanitize", "sanitizations"), ("finetune", "fine-tuning runs")):
    per_tier = {t: sum(j["tier"] == t for j in plan[key]) for t in (1, 2, 3)}
    print(f"{label}: " + ", ".join(f"tier {t}: {n}" for t, n in per_tier.items()) + f"  (running tiers <= {MAX_TIER})")

if globals().get("queue_proc") is not None and queue_proc.poll() is None:
    print("The sanitization queue is already running; not starting a second one.")
else:
    queue_proc = subprocess.Popen([sys.executable, "-u", f"{BASE}/sanitize_queue.py", f"{BASE}/plan.json"], cwd=BASE,
                                  stdout=open(f"{BASE}/logs/queue_stdout.log", "a"), stderr=subprocess.STDOUT)
    print("Sanitization queue started in the background (log: logs/queue_stdout.log)")

In [ ]:
# Step 3 · Fine-tuning on the GPU, in priority order (1.5–4 hours; resumable)
def wait_for(job_id, path):
    t0, last = time.time(), 0
    while not os.path.exists(f"{path}/.done"):
        if os.path.exists(f"{path}/.failed") or queue_proc.poll() is not None and not os.path.exists(f"{path}/.done"):
            return False
        if time.time() - last > 300:
            print(f"   waiting for the sanitization of {job_id} ({(time.time() - t0) / 60:.0f} min)", flush=True)
            last = time.time()
        time.sleep(15)
    return True


t_all, failures, finished = time.time(), 0, 0
for job in plan["finetune"]:
    if job["tier"] > MAX_TIER:
        continue
    run = job["run_name"]
    if os.path.exists(f"{BASE}/results/{run}/metrics.json"):
        print("✓ already done:", run)
        continue
    if job["needs"] and not wait_for(job["needs"], job["data_dir"]):
        print(f"✗ skipped, its sanitization failed: {run} (see logs/san__{job['needs']}.log)")
        continue
    print(f"▶ {run}  (tier {job['tier']}, {(time.time() - t_all) / 60:.0f} min so far)", flush=True)
    log = f"{BASE}/logs/ft__{run}.log"
    cmd = [sys.executable, "-u", "finetune_sanitized.py", "--data_dir", job["data_dir"], "--run_name", run, *job["args"]]
    with open(log, "w") as lf:
        code = subprocess.run(cmd, cwd=BASE, stdout=lf, stderr=subprocess.STDOUT).returncode
    lines = open(log, errors="ignore").read().splitlines()
    result = [l for l in lines if l.startswith("RESULT")]
    if code == 0 and result:
        print("  ", result[-1].replace(f"RESULT {run} | ", ""))
        finished += 1
    else:
        failures += 1
        print(f"  !! failed (exit {code}); last lines of {log}:\n     " + "\n     ".join(lines[-12:]))
        if finished == 0 and failures >= 2:
            raise RuntimeError("The first fine-tuning runs all failed. Upload the log above in the chat.")
    backup(f"{BASE}/results/{run}", log)
print(f"Fine-tuning loop finished: {finished} runs in {(time.time() - t_all) / 60:.0f} min, {failures} failed.")

In [ ]:
# Status: run this cell any time (e.g. after a reconnect) to see what has finished
st = {}
if os.path.exists(f"{BASE}/logs/queue_status.json"):
    with open(f"{BASE}/logs/queue_status.json") as f:
        st = json.load(f)
alive = globals().get("queue_proc") is not None and queue_proc.poll() is None
print(f"sanitization queue: {'running' if alive else 'not running'} | now: {st.get('current')} | "
      f"done this session: {len(st.get('done', []))} | failed: {st.get('failed', [])}")
for job in plan["finetune"]:
    if job["tier"] > MAX_TIER:
        continue
    p = f"{BASE}/results/{job['run_name']}/metrics.json"
    if os.path.exists(p):
        with open(p) as f:
            d = json.load(f)["final_dev"]
        print(f"  ✓ {job['run_name']:<42} accuracy {d['accuracy']:.4f}  macro-F1 {d['macro_f1']:.4f}")
    else:
        print(f"  · {job['run_name']}")

In [ ]:
# Step 4 · Summary tables, and one zip to upload in the chat (about 2 minutes)
import zipfile
if globals().get("queue_proc") is not None and queue_proc.poll() is None:
    print("Note: the sanitization queue is still running; the summary covers what has finished so far.\n")
!cd {BASE} && python -u analyse_runs.py --plan plan.json --out analysis

files = ["plan.json"]
files += glob.glob("analysis/**/*", recursive=True) + glob.glob("logs/*")
files += glob.glob("results/*/metrics.json") + glob.glob("results/*/dev_predictions.tsv")
files += [p for p in ("data/app_reviews/dev.tsv", "data/app_reviews/stats.json") if os.path.exists(p)]
files += glob.glob("san/**/dev.tsv", recursive=True) + glob.glob("san/**/.done", recursive=True)
with zipfile.ZipFile(f"{BASE}/final_runs_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(files)):
        if os.path.isfile(p) and "repro_dupdoc" not in p:
            z.write(p)
print(f"\nfinal_runs_outputs.zip: {os.path.getsize(f'{BASE}/final_runs_outputs.zip') / 1e6:.1f} MB")
backup(f"{BASE}/final_runs_outputs.zip", f"{BASE}/analysis")
try:
    from google.colab import files as colab_files
    colab_files.download(f"{BASE}/final_runs_outputs.zip")
except Exception as e:
    print("Download it from Drive instead (MyDrive/COMP8240_final/final_runs_outputs.zip):", e)

### Next
Upload `final_runs_outputs.zip` in the chat. If you stopped early, run Step 4 anyway: it summarises whatever has finished.